In [340]:
from openai import OpenAI
from dotenv import load_dotenv
import os
import sys
import json
from datetime import datetime
from google import genai
import numpy as np

In [341]:
from typing import get_origin, get_args, Literal

In [342]:
load_dotenv()

OPEN_ROUTER_API = os.getenv("OPENROUTER_API_KEY") or os.getenv("OPEN_ROUTER_API_KEY")
MODEL = os.getenv("OPENROUTER_MODEL") or os.getenv("MODEL")

if not OPEN_ROUTER_API:
    raise ValueError("API key not found. Please set OPENROUTER_API_KEY in .env file.")

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPEN_ROUTER_API,
)


In [343]:
class Memory:
    def __init__(self, embedding_function):
        self.data = []
        self.next_id = 1
        self.embedding_function = embedding_function

    def remember(self, key, value):
        text = f"{key} : {value}"
        embedding = self.embedding_function(text)

        for memory in self.data:

            if memory["key"] == key:
                memory["value"] = value
                memory["text"] = text
                memory["embedding"] = embedding
                memory["timestamp"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

                return

        self.data.append({
            "id" : self.next_id,
            "key": key,
            "value": value,
            "text" : f"{key} : {value}",
            "embedding": embedding,
            "timestamp" : datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        })
        
        self.next_id += 1
    
    def recall(self, key):
        for memory in reversed(self.data):
            if memory["key"] == key:
                return memory["value"]

    def forget(self, key):
        self.data = [
            memory
            for memory in self.data
            if memory["key"] != key
        ]

    def list_memories(self):
        return self.data

    def search(self, query, top_k = 3):
        query_embedding = self.embedding_function(query)

        results = []

        for memory in self.data:
            score = cosine_similarity(query_embedding, memory["embedding"])

            results.append({
                "memory" : memory,
                "score" : score
            })

        results.sort(
            key = lambda x : x["score"],
            reverse = True
        )

        return results[:top_k]

In [344]:
class Tool:
    def __init__(self, function, description):
        self.function = function
        self.description = description
        self.parameters = generate_parameters(function)

    def execute(self, arguments, context):
        try:
            if context is None:
                context = {}
            
            return self.function(**arguments, **context)
        except Exception as e:
            return f"Tool execution failed: {str(e)}"

    def schema(self):
        return {
            "type" : "function",
            "function":{
                "name": self.function.__name__,
                "description": self.description,
                "parameters": self.parameters
            }
        }
    

In [345]:
class Agent:
    def __init__(self, client, tool_list, model, system_prompt):
        self.client = client
        self.tool_list = tool_list
        self.model = model

        self.TOOL_MAP = {
            tool.function.__name__ : tool
            for tool in tool_list
        }

        self.Tool_SCHEMA = [
            tool.schema()
            for tool in tool_list
        ]

        self.messages = [{
            "role": "system",
            "content": system_prompt
        }       
        ]

        self.state = {
        }

        self.memory = Memory(create_embedding)
        
        self.context = {
            "memory" : self.memory
        }
        
    def set_state(self, key, value):
        self.state[key] = value

    def get_state(self, key):
        return self.state[key]

    def call_llm(self):
        return client.chat.completions.create(
                model=self.model,
                messages=self.messages,
                tools=self.Tool_SCHEMA,
                max_tokens=1000
            )

    def execute(self, tool_call):
            tool_name = tool_call.function.name
            print(f"TOOL CALLED: {tool_name}")

            tool = self.TOOL_MAP.get(tool_name)

            if not tool:
                return f"Tool '{tool_name}' does not exist."

            try:
                arguments = json.loads(tool_call.function.arguments)
                return tool.execute(arguments, self.context)

            except Exception as e:
                return f"Tool execution failed: {str(e)}"
        
    
    def run(self, user_input):
        self.messages.append({"role": "user", "content": user_input})

        MAX_ITERATIONS = 10

        for iteration in range(MAX_ITERATIONS):
            
            response = self.call_llm()

            response_message = response.choices[0].message
            self.messages.append(response_message)

            if not response_message.tool_calls:
                return f"AI: ",response_message.content

            for tool_call in response_message.tool_calls:
                    print("Tool is running")
                    result = self.execute(tool_call)
                    
                    self.messages.append({
                        "role": "tool",
                        "tool_call_id": tool_call.id,
                        "name": tool_call.function.name,
                        "content": json.dumps(result)
                    })

        
        else:
            print("MAX Tool iterations reached!!")
            
        

In [346]:

OPENROUTER_EMBEDDING_API_KEY = os.getenv("OPENROUTER_EMBEDDING_API_KEY")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL")

if not OPEN_ROUTER_API:
    raise ValueError("API key not found. Please set OPENROUTER_API_KEY in .env file.")

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_EMBEDDING_API_KEY,
)


In [347]:
def create_embedding(text):
    embedding = client.embeddings.create(
        model=EMBEDDING_MODEL,
        input = text,
        encoding_format="float"
        )

    return embedding.data[0].embedding    

In [348]:
def cosine_similarity(a, b):
    a = np.array(a)
    b = np.array(b)

    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [349]:
embedding1 = create_embedding(
    "favorite_language: Python"
)

embedding2 = create_embedding(
    "What programming language do I like?"
)

similarity = cosine_similarity(embedding1, embedding2)
print(similarity)

embedding3 = create_embedding(
    "I live in Vizag"
)

print(cosine_similarity(embedding1, embedding3))

0.6721677970103496
0.053729167863616256


In [350]:
def python_type_to_json_type(annotation):

    if get_origin(annotation) is Literal:

        values = get_args(annotation)

        first_value = values[0]

        if isinstance(first_value, str):
            json_type = "string"
        
        elif isinstance(first_value, int):
            json_type = "integer"

        elif isinstance(first_value, float):
            json_type = "number"
        
        elif isinstance(first_value, bool):
            json_type = "boolean"

        else:
            json_type = "string"

        return{
            "type": json_type,
            "enum": list(values)
        }
    if annotation == str:
        return "string"

    elif annotation == int:
        return "integer"

    elif annotation == float:
        return "number"

    elif annotation == bool:
        return "boolean"

    return "string"

In [351]:
import inspect

def generate_parameters(function):
    
    signature = inspect.signature(function)

    properties = {}
    required = []

    for name, parameter in signature.parameters.items():
        if name == "memory":
            continue

        json_type = python_type_to_json_type(parameter.annotation)

        properties[name] = {
            "type" : json_type
        }

        if parameter.default is inspect.Parameter.empty:
            required.append(name)

    return {
        "type" : "object",
        "properties" : properties,
        "required" : required
    }

In [352]:
def get_current_time():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

In [353]:
time_tool = Tool(
    function = get_current_time,
    description="Get current Time",
)

In [354]:
def calculator(a: float, b: float, operation: Literal["add", "subtract", "multiply", "divide"]):
    if(operation == "add"):
        return a+b

    elif(operation == "divide"):
        if(b != 0):
            return a/b
        else: return "Cannot divide with Zero"

    elif(operation == "subtract"):
        return a-b
    
    elif(operation == "multiply"):
        return a*b


In [355]:
calculator_tool = Tool(
    function=calculator,
    description="Perform mathematical calculations",
)

In [356]:
def greet(name: str, age: int, excited: bool = False):
    if excited:
        return f"Hello {name}! You are {age} years old!"
    return f"Hello {name}. You are {age} years old."

In [357]:
greet_tool = Tool(
    greet,
    "Greets a Person"
)

In [358]:
def save_memory(memory: Memory, key: str, value: str):
    memory.remember(key, value)
    return f"Remembered {key} = {value}"

In [359]:
memory_tool = Tool(
    save_memory,
    "Saves important information to agent's memory"
)

In [360]:
def recall_memory(memory: Memory, key: str):
    value = memory.recall(key)
    if value is None:
        return f"No memory found for '{key}'"

    return f"The value of {key} = {value}"

In [361]:
recall_tool = Tool(
    recall_memory,
    """Retrieve a memory using its exact key.

    Use this ONLY when you already know the exact memory key.
    For example, if the key is exactly "hometown", use:
    recall_memory(key="hometown").

    If you do not know the exact key, use search_memory instead."""
)

In [362]:
def forget_memory(memory: Memory, key: str):
    memory.forget(key)
    return f"memory forgotten"

In [363]:
forget_tool = Tool(
    forget_memory,
    "Used to forget a memory from agent's memory"
)

In [364]:
def get_all_memories(memory: Memory):
    return memory.list_memories()

In [365]:
def search_memory(memory: Memory, query: str):
    results =  memory.search(query)

    if not results:
        return f"No memories found for '{query}'"

    cleaned_results = []

    for result in results:
        memory_data = result["memory"]

        cleaned_results.append({
            "key": memory_data["key"],
            "value": memory_data["value"],
            "score": round(result["score"], 3)
        })

    return cleaned_results

In [366]:
search_memory_tool = Tool(
    search_memory,
    """Search the agent's memory using keywords when you are unsure of the
    exact memory key. Use this tool when the user asks about something
    that may be stored in memory but you do not know the exact key.

    Example:
    User asks "What programming language do I like?"
    Search using query="language".

    Do NOT use recall_memory unless you know the exact key."""
)

In [367]:
list_memory_tool = Tool(
    get_all_memories,
    "get all the memories currently stored by the agent"
)

In [374]:
tool_list = [
    calculator_tool,
    time_tool,
    greet_tool,
    memory_tool,
    forget_tool,
    search_memory_tool
]

In [375]:
agent = Agent(
    client=client,
    tool_list=tool_list,
    model = MODEL,
    system_prompt = """You are a helpful AI agent.

You have access to tools for calculations, getting the current time,
and managing memory.

Use the calculator for mathematical calculations.
Use the time tool when the user asks for the current time.

Memory rules:

1. When the user explicitly asks you to remember something,
   use save_memory.


3. If you do NOT know the exact memory key, use search_memory.
   Do not guess the key.

4. Never use get_all_memories.

5. Do not invent memories."""
)

In [376]:
while True:
    try:
        user_input = input("You: ")
    except (EOFError, KeyboardInterrupt):
        break

    if not user_input.strip():
        continue

    if user_input.lower().strip() == "exit":
        break

    try:
        response = agent.run(user_input)
        print(response)
    except Exception as e:
        print("Error:", e)


Tool is running
TOOL CALLED: save_memory
('AI: ', "Got it! I've saved that your favorite language is **Python**. 🐍 If you ever want to recall or update this information, just let me know.")
Tool is running
TOOL CALLED: save_memory
('AI: ', "Got it! I've saved that your name is **sairamesh**. I'll keep that in mind for our conversations. If you ever want to update or recall it, just let me know!")
Tool is running
TOOL CALLED: save_memory
('AI: ', "Got it! I've saved that your hometown is **Vizag**. I'll keep that in mind for future conversations. If you ever want to update or recall it, just let me know!")
Error: Error code: 429 - {'error': {'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '50', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1788566400000'}, 'limit_source': 'openrouter_free_tier_daily', 'remedy_hint': 'Wait for the daily reset (see X-RateLimit

In [371]:
agent.memory.remember("favorite_language", "Python")
agent.memory.remember("hometown", "Vizag")
agent.memory.remember("college", "ABC College")

In [372]:
results = agent.memory.search(
    "where do i live?"
)

print(results)

[{'memory': {'id': 2, 'key': 'hometown', 'value': 'Vizag', 'text': 'hometown : Vizag', 'embedding': [0.051979921758174896, 0.01270779874175787, -0.005707739852368832, -0.036472100764513016, 0.0010141109814867377, 0.019097594544291496, -0.020533503964543343, 0.037046462297439575, -0.004846194293349981, 0.02326173335313797, 0.013353957794606686, 0.041354190558195114, 0.07926219701766968, 0.027713052928447723, 0.09017511457204819, -0.018523231148719788, 0.03216437250375748, 0.047959376126527786, -0.021251458674669266, -0.01364113949239254, -0.01687193661928177, 0.01536423061043024, 0.01206163875758648, -0.022113004699349403, 0.020533503964543343, -0.049108102917671204, -0.013425753451883793, 0.03474900871515274, 0.03905673697590828, 0.060308195650577545, 0.05341583117842674, -0.03575414419174194, -0.024266868829727173, 0.06519028544425964, -0.003266694024205208, 0.025559186935424805, -0.00840007048100233, 0.08098529279232025, -0.0010275726672261953, 0.02484123222529888, -0.015651412308216

In [373]:
search_memory(
    agent.memory,
    "What programming language do I like?"
)

[{'key': 'favorite_language', 'value': 'Python', 'score': np.float64(0.678)},
 {'key': 'college', 'value': 'ABC College', 'score': np.float64(0.077)},
 {'key': 'hometown', 'value': 'Vizag', 'score': np.float64(0.048)}]